# Experimento 12 - Drop Insulin e SkinThickness, imputação por mediana e StandardScaler

## Dataset de treino

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

NORMALIZE_COLS = ['Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction']

df = pd.read_csv("../../diabetes_dataset.csv")
df.shape


### 1. Dropando `Insulin` e `SkinThickness`

65% dos registros não tinham o valor de Insulin e cerca de 40% não tinham o valor de SkinThickness, e, ao tentar imputar por mediana, criava um pico artificial enorme na coluna, então elas saem do dataset.

In [ ]:
df = df.drop(columns=['Insulin'])
df = df.drop(columns=['SkinThickness'])
df.columns.tolist()


### 2. Imputando NaN com a mediana

Guardamos as medianas calculadas aqui — elas também vão ser usadas no
dataset de previsão, pra não usar nenhuma estatística fora do treino.

In [ ]:
medianas = df.median(numeric_only=True)
df = df.fillna(medianas)
df.isna().sum()

# medianas

### 3. Normalizando com StandardScaler

Só em `Glucose`, `BloodPressure`, `BMI`, `DiabetesPedigreeFunction`.
`Pregnancies` e `Age` ficam brutas (são contagens/anos, não medidas
contínuas como as outras), e `Outcome` não é tocado. O `scaler` é ajustado
aqui e reaproveitado no dataset de previsão.

In [ ]:
scaler = StandardScaler()
df[NORMALIZE_COLS] = scaler.fit_transform(df[NORMALIZE_COLS])
df.head()


### 4. Exportando

In [ ]:
df.to_csv("diabetes-tratado-ex12.csv", index=False)
print("treino exportado com", len(df), "registros e colunas:", list(df.columns))


## Dataset de previsão (`diabetes_app.csv`)

Drop de `Insulin` e `SkinThickness` para consistência com o dataset de
treino, e usamos o mesmo `scaler` já ajustado acima (sem `fit` de novo) —
treino e previsão precisam ficar na mesma escala pro KNN funcionar.

In [ ]:
df_app = pd.read_csv("../../diabetes_app.csv")
df_app = df_app.drop(columns=['Insulin'])
df_app = df_app.drop(columns=['SkinThickness'])
df_app = df_app.fillna(medianas)
df_app[NORMALIZE_COLS] = scaler.transform(df_app[NORMALIZE_COLS])
df_app.head()


In [ ]:
df_app.to_csv("diabetes_app-tratado-ex12.csv", index=False)
print("app exportado com", len(df_app), "registros e colunas:", list(df_app.columns))


# Resultado

O servidor de previsão retornou:
```
 - Resposta do servidor:
 {"status":"success","dev_key":"Omicron","accuracy":0.6377551020408163,"old_accuracy":0.64285714285714}
```